# Church 1932：变量捕获、归约策略与史料边界

这是一个 **现代教学重建**，不是对 1932 年完整逻辑系统的复制。

- 目标：用可检查的有限例子解释变量替换、α 等价及归约策略。
- 历史边界：原文 pp.355–356 的 Rule II/III 要求 x 在 M 中出现，并对绑定变量施加避碰条件。现代允许擦除参数的 β 归约不能直接冒充原规则。
- 本实验保留常量抽象语法，但增加 `non_erasing` 转换模式；原文 p.352 本来就允许 λx[M]，没有要求 x 必须出现。
- Church 数字及其加乘是后来的教学重建，不声称来自这篇 1932 年论文。

原始文献：[Church (1932), A Set of Postulates for the Foundation of Logic](https://www.jstor.org/stable/1968337)。

## 先列问题与成功条件

1. 身份函数能否在两种策略下正确归约？
2. 替换是否保留应当自由的 y，而不是把它捕获？
3. 同一 K y Ω，改变策略或擦除许可会怎样？
4. 能否明确区分观察到循环和仅仅用尽步数？
5. 后来的数字编码能否在一个小型测试网格中得到预期结果？

有限测试不证明一般合流性、正规化定理或不可判定性。

## 运行环境与复现

只用 Python 标准库，无随机性、网络和新增依赖。请从本 notebook 所在目录启动 Python 内核。

所附输出已由 `execute_notebook.py` 从第一格到最后一格实际执行。该工具使用 Python `exec`，不是 Jupyter 内核；仅支持普通 Python 和 stdout/stderr。本 notebook 不使用 magic 或隐式富显示，也可在已有 Python 3.10+ Jupyter 内核中逐格运行。

In [1]:
import json
from pathlib import Path
import platform
import sys
sys.dont_write_bytecode = True
assert Path("experiment.py").is_file(), "请在 notebook 所在目录启动内核。"
from experiment import (
    Var, Lam, App, IDENTITY, CONSTANT, OMEGA, ADD, MULT,
    pretty, free_vars, substitute, naive_substitute,
    alpha_equivalent, debruijn, step, normalize,
    church_numeral, decode_numeral, run_experiments,
)
print("Python:", platform.python_version())
print("依赖: 标准库；策略: 强归约，允许进入 λ 体")


Python: 3.12.14
依赖: 标准库；策略: 强归约，允许进入 λ 体


## 1. 最小基线与 α 等价

绑定变量名字可以变化，自由变量名字不能随意改变。de Bruijn 键用最近绑定位置编码绑定变量，但仍保留自由变量的名字。它只判定 α 等价，不判定 β/η 可转换性。

In [2]:
for strategy in ("normal", "applicative"):
    result = normalize(App(IDENTITY, Var("a")), strategy)
    assert result.status == "normal_form" and result.term == Var("a")
    print(strategy, " -> ".join(result.trace))
assert alpha_equivalent(IDENTITY, Lam("z", Var("z")))
assert not alpha_equivalent(IDENTITY, Lam("z", Var("x")))
print("λx.x 的结构键:", debruijn(IDENTITY))
print("λz.x 的结构键:", debruijn(Lam("z", Var("x"))))


normal ((λx.x) a) -> a
applicative ((λx.x) a) -> a
λx.x 的结构键: ('lam', ('bound', 0))
λz.x 的结构键: ('lam', ('free', 'x'))


## 2. 捕获反例：替换不能只是文字粘贴

将 λy.x 中的自由 x 替换为自由 y。正确结果应与 λz.y 等价，y 仍自由；错误实现给出 λy.y，把自由变量捕获为绑定变量。代码里的 `naive_substitute` 专用于演示这个错误，正式归约器从不调用它。

In [3]:
body = Lam("y", Var("x"))
good = substitute(body, "x", Var("y"))
bad = naive_substitute(body, "x", Var("y"))
assert alpha_equivalent(good, Lam("z", Var("y")))
assert not alpha_equivalent(good, bad)
print("正确:", pretty(good), "自由变量:", sorted(free_vars(good)))
print("错误:", pretty(bad), "自由变量:", sorted(free_vars(bad)))


正确: (λy_0.y) 自由变量: ['y']
错误: (λy.y) 自由变量: []


## 3. 策略与历史限制必须分开

设 K = λa.λb.a，Ω = (λx.x x)(λx.x x)。本实验：

- `normal`：最左最外，允许进入 λ 体。
- `applicative`：最左最内，允许进入 λ 体；不是把 λ 体当作不可求值的弱 call-by-value。
- `modern`：现代允许擦除的 β 归约。
- `non_erasing`：仅当 x∈FV(M) 时收缩 (λx.M)N，自动避免捕获。

后一模式只抽出原规则的发生条件；不是完整 Rule II，更没有重建原文的真命题上下文 J、Rule III 扩张、逻辑常量及公理。

In [4]:
constant_omega = App(App(CONSTANT, Var("y")), OMEGA)
strategy_results = {}
for strategy, mode in (("normal", "modern"),
                       ("applicative", "modern"),
                       ("normal", "non_erasing")):
    result = normalize(constant_omega, strategy, mode, max_steps=16)
    strategy_results[(strategy, mode)] = result
    print(strategy, mode, result.status, "steps=", result.steps)
    print("  " + " -> ".join(result.trace))
assert strategy_results[("normal", "modern")].term == Var("y")
assert strategy_results[("applicative", "modern")].status == "cycle_detected"
assert strategy_results[("normal", "non_erasing")].status == "cycle_detected"


normal modern normal_form steps= 2
  (((λa.(λb.a)) y) ((λx.(x x)) (λx.(x x)))) -> ((λb.y) ((λx.(x x)) (λx.(x x)))) -> y
applicative modern cycle_detected steps= 2
  (((λa.(λb.a)) y) ((λx.(x x)) (λx.(x x)))) -> ((λb.y) ((λx.(x x)) (λx.(x x)))) -> ((λb.y) ((λx.(x x)) (λx.(x x))))
normal non_erasing cycle_detected steps= 2
  (((λa.(λb.a)) y) ((λx.(x x)) (λx.(x x)))) -> ((λb.y) ((λx.(x x)) (λx.(x x)))) -> ((λb.y) ((λx.(x x)) (λx.(x x))))


### 有限参数也能直接看出差别

(λx.y)z 在现代模式下一步得到 y；在非擦除模式下没有获准的归约。这里的 `normal_form` 仅指当前模式下没有可收缩位置，不代表它已是现代 β 正规形，也不说明原文能证明什么。

In [5]:
erasing_term = App(Lam("x", Var("y")), Var("z"))
modern = normalize(erasing_term)
restricted = normalize(erasing_term, mode="non_erasing")
assert modern.term == Var("y")
assert restricted.term == erasing_term and restricted.steps == 0
print("现代:", modern.status, pretty(modern.term), "steps=", modern.steps)
print("非擦除:", restricted.status, pretty(restricted.term), "steps=", restricted.steps)
# 内层同名绑定不能算作外层参数的自由发生。
shadowed = App(Lam("x", Lam("x", Var("x"))), Var("z"))
assert step(shadowed, mode="non_erasing") is None
print("内层同名绑定检查: PASS")


现代: normal_form y steps= 1
非擦除: normal_form ((λx.y) z) steps= 0
内层同名绑定检查: PASS


## 4. 循环证据与步数上限

`cycle_detected` 记录固定确定策略下真正重复的 α 等价类；Ω 的一次归约就回到自己。这是这个具体循环的证据。

`step_limit` 只说预算耗尽，结论未定。即使测试中的另一个发散项持续增长，观察八步也不能单靠超限状态证明它永不终止。程序没有把一般归约终止问题判定掉。

In [6]:
cycle = normalize(OMEGA, max_steps=8)
budget_only = normalize(OMEGA, max_steps=8, detect_cycles=False)
grow = Lam("x", App(App(Var("x"), Var("x")), Var("x")))
growth = normalize(App(grow, grow), max_steps=8)
assert cycle.status == "cycle_detected" and cycle.cycle_length == 1
assert budget_only.status == growth.status == "step_limit"
print("Ω，循环检测开启:", cycle.status, "steps=", cycle.steps, "周期=", cycle.cycle_length)
print("Ω，循环检测关闭:", budget_only.status, "steps=", budget_only.steps)
print("持续增长项:", growth.status, "steps=", growth.steps, "结论未定")


Ω，循环检测开启: cycle_detected steps= 1 周期= 1
Ω，循环检测关闭: step_limit steps= 8
持续增长项: step_limit steps= 8 结论未定


## 5. 后来的教学重建：数字、加法、乘法

数字 n 编码为 λf.λx.fⁿx。这里用现代 β 模式演示 2+3、2×3、0×3。这是理解抽象、应用如何编码计算的练习，不能写成这篇 1932 年论文已经给出了这些结果。

In [7]:
for name, operator, a, b, expected in (("add", ADD, 2, 3, 5),
                                        ("multiply", MULT, 2, 3, 6),
                                        ("multiply", MULT, 0, 3, 0)):
    result = normalize(App(App(operator, church_numeral(a)), church_numeral(b)))
    actual = decode_numeral(result.term)
    assert result.status == "normal_form" and actual == expected
    assert alpha_equivalent(result.term, church_numeral(expected))
    print(name, a, b, "=", actual, "beta steps=", result.steps)


add 2 3 = 5 beta steps= 6
multiply 2 3 = 6 beta steps= 7
multiply 0 3 = 0 beta steps= 3


## 6. 回归验证与完整结果

测试覆盖捕获、遮蔽、名字避碰、α 键、两种策略、两种规则模式、零步预算、循环，以及小范围数字加乘。自由变量性质检查遍历 507 个深度≤2的项、3个替换变量、4个替换项，共 6,084 个组合；加乘网格覆盖 0–3 与两种策略。它们仍然是有限测试。

In [8]:
import io
import unittest
import test_experiment
suite = unittest.defaultTestLoader.loadTestsFromModule(test_experiment)
test_log = io.StringIO()
summary = unittest.TextTestRunner(stream=test_log, verbosity=1).run(suite)
assert summary.wasSuccessful(), test_log.getvalue()
print("tests=", summary.testsRun, "failures=", len(summary.failures), "errors=", len(summary.errors))
report = run_experiments()
print("完整实验组:", len(report["reductions"]), "数字示例:", len(report["arithmetic"]))
print("实际命令行输出另见 results.json；详细测试日志见 tests.log。")


tests= 32 failures= 0 errors= 0
完整实验组: 10 数字示例: 3
实际命令行输出另见 results.json；详细测试日志见 tests.log。


## 结论与停止条件

已验证这组有限例子的预期行为。最重要的史料边界是：**现代允许擦除的 β 归约、原文附有发生条件的转换规则、以及后来的数字编码，必须分层陈述。**

本实验不声称证明一般合流性、1932 年体系的一致性、归约的不可判定性，也不重演 1932 年论文的全部定理。若继续扩展，应另行指定历史版本和定理的证明义务，而不是让更多通过的测试替代证明。